A binary tree becomes a **binary search tree** the moment you add one rule: for every node, all values in its left subtree are smaller, and all values in its right subtree are larger. That single constraint is enough to make search, insertion, and deletion all run in **O(h)** time — where h is the height of the tree. On a balanced tree, h is O(log n), giving you logarithmic performance on all three operations without a hash function.

BSTs are the foundation for sorted maps, sorted sets, interval trees, and the self-balancing trees (AVL, Red-Black) that back `std::map` in C++, `TreeMap` in Java/Kotlin, and `SortedDict` in Python.

## The BST Property

For every node N in a BST:
- Every value in N's **left subtree** is **less than** N's value
- Every value in N's **right subtree** is **greater than** N's value

This property holds recursively — it applies not just to direct children but to all descendants. A consequence: **in-order traversal of a BST always yields values in sorted ascending order**.

![BST Property](https://raw.githubusercontent.com/schemabotview/dsa/main/img/bst-property.svg)

## Node and Tree Setup

### Python

In [ ]:
from __future__ import annotations

class BST:
    """Binary Search Tree."""

    class Node:
        def __init__(self, val: int):
            self.val   = val
            self.left:  BST.Node | None = None
            self.right: BST.Node | None = None

    def __init__(self):
        self.root: BST.Node | None = None

# Build the tree from the diagram (insert in this order to get a balanced shape):
#        8
#       / \
#      3   10
#     / \    \
#    1   6    14
#       / \   /
#      4   7 13

bst = BST()

### Kotlin

```kotlin
data class BSTNode(val `val`: Int, var left: BSTNode? = null, var right: BSTNode? = null)

class BST {
    var root: BSTNode? = null
}
```

## Search — O(h)

At each node: if the target equals the current value, found. If the target is less, go left. If greater, go right. At most one branch is followed at each level — you discard half the remaining tree at every step.

### Python

In [ ]:
def search(node: BST.Node | None, target: int) -> BST.Node | None:
    if node is None or node.val == target:
        return node                         # found (or not in tree)
    if target < node.val:
        return search(node.left, target)    # go left
    return search(node.right, target)       # go right

# Iterative version — avoids call-stack depth
def search_iter(root: BST.Node | None, target: int) -> BST.Node | None:
    node = root
    while node:
        if target == node.val:   return node
        elif target < node.val:  node = node.left
        else:                    node = node.right
    return None

### Kotlin

```kotlin
fun search(node: BSTNode?, target: Int): BSTNode? {
    if (node == null || node.`val` == target) return node
    return if (target < node.`val`) search(node.left, target)
           else                     search(node.right, target)
}

fun searchIter(root: BSTNode?, target: Int): BSTNode? {
    var node = root
    while (node != null) {
        node = when {
            target == node.`val` -> return node
            target < node.`val`  -> node.left
            else                 -> node.right
        }
    }
    return null
}
```

## Insert — O(h)

Search for the value's correct position using the same left/right logic. When you fall off the tree (`None`), you have found the insertion point — create a new leaf there. Duplicates are typically rejected or sent right.

### Python

In [ ]:
def insert(node: BST.Node | None, val: int) -> BST.Node:
    if node is None:
        return BST.Node(val)               # insertion point — create leaf
    if val < node.val:
        node.left  = insert(node.left,  val)
    elif val > node.val:
        node.right = insert(node.right, val)
    # val == node.val → duplicate, ignore
    return node

# Build the example tree
for v in [8, 3, 10, 1, 6, 14, 4, 7, 13]:
    bst.root = insert(bst.root, v)

# In-order should give sorted output
def inorder(node):
    return [] if node is None else inorder(node.left) + [node.val] + inorder(node.right)

print(inorder(bst.root))   # [1, 3, 4, 6, 7, 8, 10, 13, 14]

### Kotlin

```kotlin
fun insert(node: BSTNode?, `val`: Int): BSTNode {
    if (node == null) return BSTNode(`val`)
    when {
        `val` < node.`val` -> node.left  = insert(node.left,  `val`)
        `val` > node.`val` -> node.right = insert(node.right, `val`)
        // duplicate — ignore
    }
    return node
}
```

## Min, Max, Successor, Predecessor

The **minimum** is always the leftmost node (go left until `None`). The **maximum** is always the rightmost node.

The **in-order successor** of node N is the smallest value greater than N — the leftmost node in N's right subtree. Used in deletion.

### Python

In [ ]:
def minimum(node: BST.Node) -> BST.Node:
    while node.left:
        node = node.left
    return node

def maximum(node: BST.Node) -> BST.Node:
    while node.right:
        node = node.right
    return node

print("Min:", minimum(bst.root).val)   # 1
print("Max:", maximum(bst.root).val)   # 14

### Kotlin

```kotlin
fun minimum(node: BSTNode): BSTNode = if (node.left == null) node else minimum(node.left!!)
fun maximum(node: BSTNode): BSTNode = if (node.right == null) node else maximum(node.right!!)
```

## Delete — O(h)

Deletion has three cases depending on the target node's children:

1. **Leaf (no children)** — simply remove it (return `None` to the parent).
2. **One child** — replace the node with its only child.
3. **Two children** — find the **in-order successor** (minimum of the right subtree), copy its value into the current node, then delete the successor from the right subtree. The successor always has at most one child (it has no left child by definition), so the deletion reduces to case 1 or 2.

![BST Deletion Cases](https://raw.githubusercontent.com/schemabotview/dsa/main/img/bst-delete-cases.svg)

### Python

In [ ]:
def delete(node: BST.Node | None, val: int) -> BST.Node | None:
    if node is None:
        return None                          # value not found

    if val < node.val:
        node.left  = delete(node.left,  val) # search left
    elif val > node.val:
        node.right = delete(node.right, val) # search right
    else:
        # Found the node to delete
        if node.left is None:                # case 1 or 2 — no left child
            return node.right
        if node.right is None:               # case 2 — no right child
            return node.left
        # Case 3 — two children: replace with in-order successor
        successor   = minimum(node.right)
        node.val    = successor.val
        node.right  = delete(node.right, successor.val)

    return node

# Delete 3 (two children: successor is 4)
bst.root = delete(bst.root, 3)
print(inorder(bst.root))   # [1, 4, 6, 7, 8, 10, 13, 14]

# Delete 14 (one child: 13 promoted)
bst.root = delete(bst.root, 14)
print(inorder(bst.root))   # [1, 4, 6, 7, 8, 10, 13]

### Kotlin

```kotlin
fun delete(node: BSTNode?, `val`: Int): BSTNode? {
    if (node == null) return null
    when {
        `val` < node.`val` -> node.left  = delete(node.left,  `val`)
        `val` > node.`val` -> node.right = delete(node.right, `val`)
        else -> {
            if (node.left  == null) return node.right
            if (node.right == null) return node.left
            val successor  = minimum(node.right!!)
            node.`val`     = successor.`val`
            node.right     = delete(node.right, successor.`val`)
        }
    }
    return node
}
```

## Validate a BST

The naive check — "left child < node < right child" — is wrong. It misses cases where a deep descendant violates the global ordering (e.g., a node in the left subtree is larger than the root). The correct approach threads a valid range `[min, max]` down the tree.

### Python

In [ ]:
def is_valid_bst(node: BST.Node | None,
                 lo: float = float('-inf'),
                 hi: float = float('inf')) -> bool:
    if node is None:
        return True
    if not (lo < node.val < hi):
        return False
    return (is_valid_bst(node.left,  lo,       node.val) and
            is_valid_bst(node.right, node.val, hi))

print(is_valid_bst(bst.root))   # True

# Construct an invalid BST (violates property at a deep node)
from __future__ import annotations
bad = BST.Node(5)
bad.left  = BST.Node(3)
bad.right = BST.Node(8)
bad.left.right = BST.Node(7)    # 7 > 5 but lives in left subtree — invalid!
print(is_valid_bst(bad))        # False

### Kotlin

```kotlin
fun isValidBST(
    node: BSTNode?,
    lo: Long = Long.MIN_VALUE,
    hi: Long = Long.MAX_VALUE
): Boolean {
    if (node == null) return true
    if (node.`val` <= lo || node.`val` >= hi) return false
    return isValidBST(node.left,  lo,             node.`val`.toLong()) &&
           isValidBST(node.right, node.`val`.toLong(), hi)
}
```

## Build a Balanced BST from a Sorted Array

Given a sorted array, the optimal strategy is to pick the middle element as the root, recurse on the left half for the left subtree, and recurse on the right half for the right subtree. This produces a perfectly balanced BST of height ⌊log₂ n⌋.

### Python

In [ ]:
def sorted_array_to_bst(nums: list[int]) -> BST.Node | None:
    if not nums:
        return None
    mid  = len(nums) // 2
    node = BST.Node(nums[mid])
    node.left  = sorted_array_to_bst(nums[:mid])
    node.right = sorted_array_to_bst(nums[mid + 1:])
    return node

balanced = sorted_array_to_bst([1, 2, 3, 4, 5, 6, 7])
print(inorder(balanced))            # [1, 2, 3, 4, 5, 6, 7]

def height(node):
    return -1 if node is None else 1 + max(height(node.left), height(node.right))

print("Height:", height(balanced))  # 2  (perfect for 7 nodes)

### Kotlin

```kotlin
fun sortedArrayToBST(nums: List<Int>): BSTNode? {
    if (nums.isEmpty()) return null
    val mid  = nums.size / 2
    val node = BSTNode(nums[mid])
    node.left  = sortedArrayToBST(nums.subList(0, mid))
    node.right = sortedArrayToBST(nums.subList(mid + 1, nums.size))
    return node
}
```

## BST Performance

All BST operations are O(h) — proportional to the height.

| Tree shape | Height | Search / Insert / Delete |
|---|---|---|
| Perfectly balanced | O(log n) | **O(log n)** |
| Random insertions (average) | O(log n) | O(log n) |
| Sorted insertions (worst case) | O(n) | **O(n)** — degenerates to a linked list |

Inserting values `1, 2, 3, 4, 5` into a plain BST in sorted order produces a right-skewed chain — every node has only a right child. The height becomes n, and the BST is no faster than a linked list.

**Self-balancing BSTs** (AVL trees, Red-Black trees) automatically rebalance after insertions and deletions, keeping the height at O(log n) in all cases. These are what power `TreeMap` in Kotlin/Java and `SortedDict` in Python's `sortedcontainers` library.

## Key Takeaways

- The **BST property**: for every node, all values in its left subtree are smaller and all values in its right subtree are larger. This holds recursively at every level.
- **In-order traversal** of a BST always yields values in sorted ascending order — a direct consequence of the ordering property.
- **Search, insert, and delete** are all O(h). On a balanced tree h = O(log n); on a degenerate (sorted-insertion) tree h = O(n).
- **Minimum** is the leftmost node; **maximum** is the rightmost. The **in-order successor** is the minimum of a node's right subtree.
- **Deletion** has three cases: leaf (remove), one child (promote child), two children (replace with in-order successor and delete the successor).
- **BST validation** requires threading a valid range through the recursion — checking only immediate children misses global violations.
- **Building from a sorted array**: pick the midpoint as root, recurse on halves — produces a perfectly balanced tree in O(n) time.
- When you need guaranteed O(log n) performance, use a **self-balancing BST** (`TreeMap` in Kotlin, `sortedcontainers.SortedDict` in Python).